In [1]:
from pathlib import Path
import os
import sys

os.environ.setdefault("AGRIWATER_DB", r"C:\Users\menna\Downloads\agriwater_v2.sqlite")

HERE = Path.cwd().resolve()
ML_ROOT = next((p for p in (HERE, *HERE.parents)
                if (p / "src" / "data_cleaning.py").is_file()), None)
if ML_ROOT is None:
    for f in (Path.home() / "Downloads").rglob("data_cleaning.py"):
        if f.parent.name == "src":
            ML_ROOT = f.parent.parent
            break
if ML_ROOT is None:
    raise RuntimeError("Could not locate machine_learning/src/data_cleaning.py")

# ---- auto-fix 1: create src/data_loader.py if missing ----
SRC = ML_ROOT / "src"
(SRC / "__init__.py").touch(exist_ok=True)
if not (SRC / "data_loader.py").is_file():
    (SRC / "data_loader.py").write_text('''from pathlib import Path


def _is_sqlite(path):
    try:
        with path.open("rb") as f:
            return f.read(16) == b"SQLite format 3\\x00"
    except OSError:
        return False


def resolve_database(configured=None, root=None):
    candidates = []
    if configured:
        candidates.append(Path(configured).expanduser().resolve())
    if root is not None:
        root = Path(root)
        candidates += [root / "AgriWater" / "database" / "agriwater_v2.sqlite",
                       root / "database" / "agriwater_v2.sqlite"]
    for c in candidates:
        if c.exists() and c.stat().st_size > 1_000_000 and _is_sqlite(c):
            return c
    raise RuntimeError("Full agriwater_v2.sqlite not found. Set AGRIWATER_DB to the full file.")
''', encoding="utf-8")

# ---- auto-fix 2 + 3: bugs in data_cleaning.py ----
dc = SRC / "data_cleaning.py"
txt = dc.read_text(encoding="utf-8")

old2 = "bad = int(((s < lo) if lo is not None else False).sum() + ((s > hi) if hi is not None else False).sum())"
new2 = "bad = int((s < lo).sum() if lo is not None else 0) + int((s > hi).sum() if hi is not None else 0)"
txt = txt.replace(old2, new2)

old3 = 'outlier_rows("soil_point", sp, list(SOIL_DOMAIN), SOIL_DOMAIN,'
new3 = ('outlier_rows("soil_point", sp, '
        '[{"bulk": "bulk_density", "ph_water": "ph"}.get(k, k) for k in SOIL_DOMAIN], '
        '{{"bulk": "bulk_density", "ph_water": "ph"}.get(k, k): v for k, v in SOIL_DOMAIN.items()},')
txt = txt.replace(old3, new3)

dc.write_text(txt, encoding="utf-8")

sys.path.insert(0, str(ML_ROOT))
sys.path.insert(0, str(ML_ROOT.parent))
for m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[m]
print("Machine-learning folder:", ML_ROOT)

from src.data_loader import resolve_database
from src.data_cleaning import run
db = resolve_database(os.environ.get("AGRIWATER_DB"), ML_ROOT.parent)
run(db, ML_ROOT)

Machine-learning folder: C:\Users\menna\Downloads\Smart-Agricultural-Water-Management-main (1)\Smart-Agricultural-Water-Management-main\machine_learning
                                            Item  Value
                              Tables in database     44
                       Non-empty tables profiled     34
                                    Empty tables     10
                      Cleaning operations logged     22
                             Validity checks run     44
                           Checks needing review      3
                   Weather records (point x day) 136230
         Distinct weather series among 38 points      5
          Rainfall point-days flagged (> 150 mm)     67
Exact duplicate rows removed (database_metadata)     10
                             Dataset                                                                                          Check  Records failing                                                                                   